# MAWRID Benchmark — 3-Stage Pipeline
**Stage 1** Vision → raw OCR text | **Stage 2** Text → doc type | **Stage 3** Text → fields

Models: **Claude** (ground truth) · **Groq** (llama-4-scout) · **Gemma3** (local HF)

Upload to `/gdrive/MyDrive/mawrid_data/`: `schema_v2.json` and `raw_pdfs/` folder.

## 1 — Installs

In [ ]:
!pip install pdf2image==1.17.0 "pillow==12.1.0" --force-reinstall -q
!apt-get install -y poppler-utils -q
!pip install groq anthropic -q
!pip install transformers accelerate bitsandbytes -q

In [ ]:
# Restart runtime so the freshly installed Pillow is picked up.
# Everything above this cell re-runs automatically after restart.
import os
os.kill(os.getpid(), 9)

## 2 — Mount Drive & Keys

In [ ]:
import os, json, time, base64, re
from os.path import join
from glob import glob
from tqdm.auto import tqdm
from difflib import SequenceMatcher
from pdf2image import convert_from_path
from PIL import Image, ImageEnhance
from google.colab import userdata, drive

drive.mount('/gdrive')

data_dir = "/gdrive/MyDrive/mawrid_data"
GT_PATH  = f"{data_dir}/ground_truth.json"

os.environ["GROQ_API_KEY"]      = userdata.get('groq-key')
os.environ["ANTHROPIC_API_KEY"] = userdata.get('anthropic-key')

print("Keys loaded ✓")

## 3 — Load Schema

In [ ]:
with open(f"{data_dir}/schema_v2.json", encoding="utf-8") as f:
    SCHEMA = json.load(f)

DOCUMENTS = SCHEMA["documents"]

DOC_TYPES_LIST = "\n".join(
    f"  {key} — {doc['label_ar']}"
    for key, doc in DOCUMENTS.items()
)

SCHEMA_FIELDS = {
    key: [f["label_ar"] for f in doc.get("fields", [])]
    for key, doc in DOCUMENTS.items()
}

# field name → type hint for the extraction template
SCHEMA_FIELD_TYPES = {
    key: {f["label_ar"]: f.get("type", "text") for f in doc.get("fields", [])}
    for key, doc in DOCUMENTS.items()
}

print(f"Schema loaded ✓  ({len(DOCUMENTS)} document types)")

## 4 — PDF → Images

In [ ]:
def preprocess_image(image, maxwidth=800):
    gray = image.convert('L')
    if gray.width > maxwidth:
        h = int(gray.height * maxwidth / gray.width)
        gray = gray.resize((maxwidth, h), Image.LANCZOS)
    return ImageEnhance.Contrast(gray).enhance(1.5)

def convert_pdf_to_images(pdf_path, output_dir, maxwidth=800):
    name = os.path.basename(pdf_path).replace('.pdf', '')
    out  = join(output_dir, name)
    os.makedirs(out, exist_ok=True)
    paths = []
    for i, img in enumerate(convert_from_path(pdf_path, dpi=200), 1):
        p = join(out, f"{name}_page_{i:03d}.jpeg")
        preprocess_image(img, maxwidth).save(p, 'JPEG', quality=85, optimize=True)
        paths.append(p)
    return paths

pdf_files = glob(f"{data_dir}/raw_pdfs/*.pdf")
if pdf_files:
    for pdf in tqdm(pdf_files, desc="Converting PDFs"):
        convert_pdf_to_images(pdf, f"{data_dir}/processed_images")
    print(f"Done — {len(pdf_files)} PDFs")
else:
    print("No PDFs found — skipping")

## 5 — Prompts

In [ ]:
SYSTEM_OCR = (
    "You are an OCR engine that transcribes Arabic official documents exactly as written — "
    "no paraphrasing, no translation, no summarization."
)

PROMPT_OCR = """Extract ALL visible text from this document image exactly as it appears.

Rules:
- Arabic stays Arabic, English stays English — do NOT romanize or translate
- Copy letter by letter — dots matter (ي/ن, ز/ر, ث/ت/ب)
- Include headers, body, footer, stamps, handwritten annotations, all reference numbers and dates
- If illegible write [unclear]

Return plain text only."""

_CLASSIFY_TEMPLATE = """You are given the raw OCR text of an Arabic official document.
Identify the document type. Reply with ONLY a JSON object:
{{"doc_type": "<key>", "confidence": 0.0}}

The value of doc_type MUST be one of these exact keys:
{doc_types_list}

OCR TEXT:
{raw_text}"""

def build_classify_prompt(raw_text: str) -> str:
    return _CLASSIFY_TEMPLATE.format(
        doc_types_list=DOC_TYPES_LIST,
        raw_text=raw_text
    )

_EXTRACT_TEMPLATE = """Extract fields from this Arabic document by filling in the JSON template below.

Document type: {doc_type}

JSON template to fill (replace each placeholder with the value from the text, keep null if not found):
{json_template}

Rules:
- Output ONLY the filled JSON — no extra keys, no explanation
- Do NOT rename or add any keys
- For date placeholders (DD/MM/YYYY): always return numeric format like 28/06/2000, never words
- Values must come from the OCR text below

OCR text:
{raw_text}"""

def _field_placeholder(field_name: str, field_type: str) -> object:
    """Return a format hint that guides the model toward the correct data type."""
    if field_type == "date" or "تاريخ" in field_name:
        return "DD/MM/YYYY"
    if field_type == "number":
        return 0
    return None

def build_extract_prompt(raw_text: str, doc_type: str) -> str:
    expected   = SCHEMA_FIELDS.get(doc_type, [])
    field_types = SCHEMA_FIELD_TYPES.get(doc_type, {})
    if expected:
        template = json.dumps(
            {f: _field_placeholder(f, field_types.get(f, "text")) for f in expected},
            ensure_ascii=False, indent=2
        )
    else:
        template = json.dumps({"الحقول": "استخرج كل الحقول الموجودة"}, ensure_ascii=False)
    return _EXTRACT_TEMPLATE.format(
        doc_type=doc_type, json_template=template, raw_text=raw_text
    )

print("Prompts loaded ✓")

## 6 — Load Gemma3 Model (needs GPU)

In [ ]:
from google.colab import userdata
hf_token = userdata.get('my-hf')

!hf auth login --token {hf_token}

In [ ]:
import torch
from transformers import AutoProcessor, Gemma3ForConditionalGeneration
from google.colab import userdata as _ud


GEMMA_MODEL_ID = "google/gemma-3-4b-it"

gemma_model = Gemma3ForConditionalGeneration.from_pretrained(
    GEMMA_MODEL_ID, device_map="auto", torch_dtype="auto"
).eval()

gemma_processor = AutoProcessor.from_pretrained(GEMMA_MODEL_ID)

print(f"Gemma3 loaded ✓  (device: {next(gemma_model.parameters()).device})")

## 7 — Stage Functions (Claude · Groq · Gemma3)

In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────────

def _to_b64(path: str) -> str:
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()

def _strip_fence(text: str) -> str:
    return re.sub(r'^```(?:json)?\s*|\s*```$', '', text.strip(), flags=re.DOTALL).strip()

def _parse_json(text: str) -> dict:
    text = _strip_fence(text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return {}

def _map_doc_type(key: str) -> str:
    # 1. exact key match
    if key in DOCUMENTS:
        return key
    # 2. label_ar or underscore-to-space match
    for k, doc in DOCUMENTS.items():
        if doc["label_ar"] == key or k.replace("_", " ") == key:
            return k
    # 3. fuzzy fallback — catches slight variations like شهادة_ميلاد vs شهادة_الميلاد
    best_k, best_score = None, 0.0
    for k in DOCUMENTS:
        score = SequenceMatcher(None, key, k).ratio()
        if score > best_score:
            best_k, best_score = k, score
    if best_score >= 0.8:
        return best_k
    return key

def _gemma_text_messages(text: str) -> list:
    return [{"role": "user", "content": [{"type": "text", "text": text}]}]


# ── Stage 1 : OCR (vision) ────────────────────────────────────────────────────

def stage1_claude(image_path: str) -> tuple:
    import anthropic
    t0  = time.time()
    msg = anthropic.Anthropic().messages.create(
        model="claude-sonnet-4-6", max_tokens=1500, system=SYSTEM_OCR,
        messages=[{"role": "user", "content": [
            {"type": "image", "source": {
                "type": "base64", "media_type": "image/jpeg", "data": _to_b64(image_path)
            }},
            {"type": "text", "text": PROMPT_OCR}
        ]}]
    )
    return msg.content[0].text.strip(), round(time.time() - t0, 2)

def stage1_groq(image_path: str) -> tuple:
    from groq import Groq
    t0 = time.time()
    resp = Groq().chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{"role": "user", "content": [
            {"type": "image_url",
             "image_url": {"url": f"data:image/jpeg;base64,{_to_b64(image_path)}"}},
            {"type": "text", "text": SYSTEM_OCR + "\n\n" + PROMPT_OCR}
        ]}],
        max_tokens=1500,
    )
    return resp.choices[0].message.content.strip(), round(time.time() - t0, 2)

def stage1_gemma(image_path: str) -> tuple:
    # system role with a plain string crashes apply_chat_template — merge into user instead
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image_path},
        {"type": "text",  "text": SYSTEM_OCR + "\n\n" + PROMPT_OCR}
    ]}]
    inputs = gemma_processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(gemma_model.device)
    input_len = inputs["input_ids"].shape[-1]
    t0 = time.time()
    with torch.inference_mode():
        gen = gemma_model.generate(**inputs, max_new_tokens=1024, do_sample=False)
    text = gemma_processor.decode(gen[0][input_len:], skip_special_tokens=True)
    return text.strip(), round(time.time() - t0, 2)


# ── Stage 2 : Classify ────────────────────────────────────────────────────────

def stage2_claude(raw_text: str) -> tuple:
    import anthropic
    t0  = time.time()
    msg = anthropic.Anthropic().messages.create(
        model="claude-sonnet-4-6", max_tokens=150,
        messages=[{"role": "user", "content": build_classify_prompt(raw_text)}]
    )
    result = _parse_json(msg.content[0].text)
    result["doc_type"] = _map_doc_type(result.get("doc_type", "unknown"))
    return result, round(time.time() - t0, 2)

def stage2_groq(raw_text: str) -> tuple:
    from groq import Groq
    t0 = time.time()
    resp = Groq().chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{"role": "user", "content": build_classify_prompt(raw_text)}],
        max_tokens=150, response_format={"type": "json_object"},
    )
    result = json.loads(resp.choices[0].message.content)
    result["doc_type"] = _map_doc_type(result.get("doc_type", "unknown"))
    return result, round(time.time() - t0, 2)

def stage2_gemma(raw_text: str) -> tuple:
    messages = _gemma_text_messages(build_classify_prompt(raw_text))
    inputs = gemma_processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(gemma_model.device)
    input_len = inputs["input_ids"].shape[-1]
    t0 = time.time()
    with torch.inference_mode():
        gen = gemma_model.generate(**inputs, max_new_tokens=150, do_sample=False)
    raw = gemma_processor.decode(gen[0][input_len:], skip_special_tokens=True)
    result = _parse_json(raw)
    result["doc_type"] = _map_doc_type(result.get("doc_type", "unknown"))
    return result, round(time.time() - t0, 2)


# ── Stage 3 : Extract ────────────────────────────────────────────────────────

def stage3_claude(raw_text: str, doc_type: str) -> tuple:
    import anthropic
    t0  = time.time()
    msg = anthropic.Anthropic().messages.create(
        model="claude-sonnet-4-6", max_tokens=800,
        messages=[{"role": "user", "content": build_extract_prompt(raw_text, doc_type)}]
    )
    return _parse_json(msg.content[0].text), round(time.time() - t0, 2)

def stage3_groq(raw_text: str, doc_type: str) -> tuple:
    from groq import Groq
    t0 = time.time()
    resp = Groq().chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{"role": "user", "content": build_extract_prompt(raw_text, doc_type)}],
        max_tokens=800, response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content), round(time.time() - t0, 2)

def stage3_gemma(raw_text: str, doc_type: str) -> tuple:
    messages = _gemma_text_messages(build_extract_prompt(raw_text, doc_type))
    inputs = gemma_processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(gemma_model.device)
    input_len = inputs["input_ids"].shape[-1]
    t0 = time.time()
    with torch.inference_mode():
        gen = gemma_model.generate(**inputs, max_new_tokens=800, do_sample=False)
    raw = gemma_processor.decode(gen[0][input_len:], skip_special_tokens=True)
    return _parse_json(raw), round(time.time() - t0, 2)


print("All stage functions loaded ✓  (claude · groq · gemma3)")

## 8 — Pipeline Runner

In [ ]:
STAGE_FNS = {
    "claude": (stage1_claude, stage2_claude, stage3_claude),
    "groq":   (stage1_groq,   stage2_groq,   stage3_groq),
    "gemma":  (stage1_gemma,  stage2_gemma,  stage3_gemma),
}

def run_pipeline(image_paths: list, doc_id: str, provider: str) -> dict:
    s1, s2, s3 = STAGE_FNS[provider]

    pages_text, t1_total = [], 0.0
    for path in image_paths:
        text, t = s1(path)
        pages_text.append(text)
        t1_total += t
    raw_text = "\n\n--- PAGE BREAK ---\n\n".join(pages_text)

    cls_result, t2 = s2(raw_text)
    doc_type        = cls_result.get("doc_type", "unknown")

    fields, t3 = s3(raw_text, doc_type)

    return {
        "doc_id":     doc_id,
        "provider":   provider,
        "raw_text":   raw_text,
        "doc_type":   doc_type,
        "confidence": cls_result.get("confidence"),
        "fields":     fields,
        "timing": {
            "stage1": round(t1_total, 2),
            "stage2": round(t2, 2),
            "stage3": round(t3, 2),
            "total":  round(t1_total + t2 + t3, 2),
        },
    }

print("Pipeline runner loaded ✓")

## 9 — Smoke Test (one image, all 3 models)
Run this **before** generating ground truth to verify all providers work correctly.

In [ ]:
SMOKE_IMAGE  = "/gdrive/MyDrive/mawrid_data/processed_images/0001/0001_page_001.jpeg"
smoke_doc_id = os.path.basename(os.path.dirname(SMOKE_IMAGE))   # → "0001"
smoke_results = {}
print(f"Smoke image : {SMOKE_IMAGE}")
print(f"Smoke doc   : {smoke_doc_id}")

In [ ]:
# ── Claude ────────────────────────────────────────────────────────────────────
try:
    result = run_pipeline([SMOKE_IMAGE], smoke_doc_id, provider="claude")
    smoke_results["claude"] = result
    print(f"Stage 1 OCR ({result['timing']['stage1']}s):\n{result['raw_text'][:400]}")
    print(f"\nStage 2 Doc Type ({result['timing']['stage2']}s): {result['doc_type']}")
    print(f"Stage 3 Fields ({result['timing']['stage3']}s):")
    for k, v in result["fields"].items():
        print(f"  {k}: {v}")
except Exception as e:
    import traceback; traceback.print_exc()
    smoke_results["claude"] = None

In [ ]:
# ── Groq ──────────────────────────────────────────────────────────────────────
try:
    result = run_pipeline([SMOKE_IMAGE], smoke_doc_id, provider="groq")
    smoke_results["groq"] = result
    print(f"Stage 1 OCR ({result['timing']['stage1']}s):\n{result['raw_text'][:400]}")
    print(f"\nStage 2 Doc Type ({result['timing']['stage2']}s): {result['doc_type']}")
    print(f"Stage 3 Fields ({result['timing']['stage3']}s):")
    for k, v in result["fields"].items():
        print(f"  {k}: {v}")
except Exception as e:
    import traceback; traceback.print_exc()
    smoke_results["groq"] = None

In [ ]:
# ── Gemma3 ────────────────────────────────────────────────────────────────────
try:
    result = run_pipeline([SMOKE_IMAGE], smoke_doc_id, provider="gemma")
    smoke_results["gemma"] = result
    print(f"Stage 1 OCR ({result['timing']['stage1']}s):\n{result['raw_text'][:400]}")
    print(f"\nStage 2 Doc Type ({result['timing']['stage2']}s): {result['doc_type']}")
    print(f"Stage 3 Fields ({result['timing']['stage3']}s):")
    for k, v in result["fields"].items():
        print(f"  {k}: {v}")
except Exception as e:
    import traceback; traceback.print_exc()
    smoke_results["gemma"] = None

In [ ]:
# ── Summary + Save ────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print("  Smoke Test Summary")
print(f"{'='*55}")
print(f"{'Provider':<12} {'Doc Type':<30} {'Fields':>6} {'Time':>7}")
print("-" * 55)
for p, r in smoke_results.items():
    if r:
        print(f"{p:<12} {r['doc_type']:<30} {len(r['fields']):>6} {r['timing']['total']:>6.1f}s")
    else:
        print(f"{p:<12} {'FAILED':<30}")

SMOKE_PATH = f"{data_dir}/smoke_test_result.json"
save_data = {
    "image":   SMOKE_IMAGE,
    "doc_id":  smoke_doc_id,
    "run_at":  time.strftime("%Y-%m-%d %H:%M:%S"),
    "providers": {}
}
for p, r in smoke_results.items():
    if r:
        save_data["providers"][p] = {
            "doc_type":      r["doc_type"],
            "confidence":    r["confidence"],
            "timing":        r["timing"],
            "stage1_ocr":    r["raw_text"],
            "stage2_cls":    r["doc_type"],
            "stage3_fields": r["fields"],
        }
    else:
        save_data["providers"][p] = {"error": "FAILED"}

with open(SMOKE_PATH, "w", encoding="utf-8") as f:
    json.dump(save_data, f, ensure_ascii=False, indent=2)
print(f"\nFull results saved → {SMOKE_PATH}")

In [ ]:
def generate_ground_truth(num_docs: int = 15):
    image_root = f"{data_dir}/processed_images"
    doc_dirs   = sorted(glob(f"{image_root}/*/"))[:num_docs]

    gt = {}
    if os.path.exists(GT_PATH):
        with open(GT_PATH, encoding="utf-8") as f:
            gt = json.load(f)
        print(f"Resuming — {len(gt)} docs already done")

    for doc_dir in tqdm(doc_dirs, desc="Claude Ground Truth"):
        doc_id = os.path.basename(doc_dir.rstrip("/"))
        if doc_id in gt:
            print(f"  skip {doc_id}")
            continue

        pages  = sorted(glob(f"{doc_dir}*.jpeg"))
        result = run_pipeline(pages, doc_id, provider="claude")

        gt[doc_id] = {
            "doc_type":       result["doc_type"],
            "reference_text": result["raw_text"],
            "fields":         result["fields"],
            "num_pages":      len(pages),
            "claude_model":   "claude-sonnet-4-6",
            "generated_at":   "2026-05-19",
        }
        with open(GT_PATH, "w", encoding="utf-8") as f:
            json.dump(gt, f, ensure_ascii=False, indent=2)

        t = result["timing"]["total"]
        print(f"  ✓ {doc_id}: {result['doc_type']} | {len(pages)}p | {t:.1f}s")

    print(f"\nGround truth ready → {GT_PATH}  ({len(gt)} docs)")
    return gt


# Uncomment, run once, then recomment:
# gt = generate_ground_truth(num_docs=15)

## 11 — Scorer

In [ ]:
def normalize_arabic(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)  # tashkeel
    text = re.sub(r'[أإآ]', 'ا', text)                  # alef
    text = text.replace('ة', 'ه').replace('ى', 'ي')     # ta marbuta + ya
    text = text.replace('_', ' ')                        # schema underscores
    return text.strip()

def score_doc(gt_doc: dict, pred: dict) -> dict:
    gt_f   = {normalize_arabic(k): normalize_arabic(str(v))
               for k, v in gt_doc.get("fields", {}).items() if v is not None}
    pred_f = {normalize_arabic(k): normalize_arabic(str(v))
               for k, v in pred.get("fields", {}).items() if v is not None}

    matched   = set(gt_f) & set(pred_f)
    precision = len(matched) / len(pred_f) if pred_f else 0.0
    recall    = len(matched) / len(gt_f)   if gt_f   else 0.0
    f1 = (2 * precision * recall / (precision + recall)
          if precision + recall > 0 else 0.0)

    correct_vals = sum(
        1 for k in matched
        if SequenceMatcher(None, gt_f[k], pred_f[k]).ratio() >= 0.85
    )
    val_acc = correct_vals / len(matched) if matched else 0.0

    return {
        "cls_correct":    int(normalize_arabic(pred.get("doc_type", "")) ==
                              normalize_arabic(gt_doc.get("doc_type", ""))),
        "precision":      round(precision, 3),
        "recall":         round(recall, 3),
        "f1":             round(f1, 3),
        "value_accuracy": round(val_acc, 3),
        "matched_keys":   len(matched),
    }

print("Scorer loaded ✓")

## 12 — Benchmark Runner

In [ ]:
from IPython.display import display, HTML

def view_ground_truth(gt_path: str = GT_PATH, ocr_preview_chars: int = 400):
    with open(gt_path, encoding="utf-8") as f:
        gt = json.load(f)

    cards = []
    for doc_id, doc in sorted(gt.items()):
        ocr_preview = doc.get("reference_text", "")[:ocr_preview_chars].replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br>")
        if len(doc.get("reference_text", "")) > ocr_preview_chars:
            ocr_preview += "<br><em>…</em>"

        fields_rows = "".join(
            f"<tr><td style='padding:4px 10px;border-bottom:1px solid #eee;color:#555'>{k}</td>"
            f"<td style='padding:4px 10px;border-bottom:1px solid #eee;font-weight:500'>"
            f"{'<span style=\"color:#aaa\">null</span>' if v is None else v}</td></tr>"
            for k, v in doc.get("fields", {}).items()
        )

        doc_label = DOCUMENTS.get(doc["doc_type"], {}).get("label_ar", doc["doc_type"])

        cards.append(f"""
        <div style='border:1px solid #ddd;border-radius:8px;margin:16px 0;font-family:Arial;direction:rtl'>
          <div style='background:#2c3e50;color:white;padding:10px 16px;border-radius:7px 7px 0 0;display:flex;justify-content:space-between;align-items:center'>
            <span style='font-size:16px;font-weight:bold'>وثيقة {doc_id}</span>
            <span style='font-size:13px;opacity:.8'>{doc.get("num_pages",1)} صفحة</span>
          </div>

          <div style='padding:12px 16px;background:#f9f9f9;border-bottom:1px solid #eee'>
            <span style='font-size:11px;color:#888;text-transform:uppercase'>المرحلة 2 — نوع الوثيقة</span><br>
            <span style='font-size:15px;font-weight:bold;color:#2980b9'>{doc["doc_type"]}</span>
            <span style='color:#888;margin-right:8px'>({doc_label})</span>
          </div>

          <div style='padding:12px 16px;border-bottom:1px solid #eee'>
            <span style='font-size:11px;color:#888;text-transform:uppercase'>المرحلة 1 — نص OCR</span><br>
            <div style='margin-top:6px;font-size:13px;line-height:1.6;color:#333;background:#fff;padding:8px;border-radius:4px;border:1px solid #eee;max-height:180px;overflow-y:auto'>
              {ocr_preview}
            </div>
          </div>

          <div style='padding:12px 16px'>
            <span style='font-size:11px;color:#888;text-transform:uppercase'>المرحلة 3 — الحقول المستخرجة</span><br>
            <table style='width:100%;margin-top:8px;border-collapse:collapse;font-size:13px'>
              {fields_rows if fields_rows else "<tr><td style='color:#aaa'>لا توجد حقول</td></tr>"}
            </table>
          </div>
        </div>
        """)

    display(HTML(f"<div style='max-width:900px'>{''.join(cards)}</div>"))
    print(f"✓ {len(gt)} وثيقة")

view_ground_truth()

## 10c — Human Ground Truth Editor
Pre-loaded from `ground_truth.json`. Fix doc type + fields, then **Save All**.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

HUMAN_GT_PATH = f"{data_dir}/human_gt.json"

# ── Load starting data ────────────────────────────────────────────────────────
with open(GT_PATH, encoding="utf-8") as f:
    _source_gt = json.load(f)

if os.path.exists(HUMAN_GT_PATH):
    with open(HUMAN_GT_PATH, encoding="utf-8") as f:
        human_gt = json.load(f)
    print(f"Resumed human_gt.json ({len(human_gt)} docs)")
else:
    human_gt = {
        doc_id: {"doc_type": doc["doc_type"], "fields": dict(doc["fields"])}
        for doc_id, doc in _source_gt.items()
    }
    print(f"Initialized from ground_truth.json ({len(human_gt)} docs)")

# ── State ─────────────────────────────────────────────────────────────────────
_field_widgets = {}   # field_name → Text widget for the currently shown doc

# ── Widgets ───────────────────────────────────────────────────────────────────
doc_opts = sorted(human_gt.keys())

doc_dd = widgets.Dropdown(
    options=doc_opts, description="وثيقة:",
    layout=widgets.Layout(width="160px")
)
type_dd = widgets.Dropdown(
    options=sorted(DOCUMENTS.keys()), description="نوع:",
    layout=widgets.Layout(width="420px")
)
fields_out  = widgets.Output()
status_lbl  = widgets.Label(value="")
null_lbl    = widgets.Label(value="")

save_doc_btn = widgets.Button(
    description="💾 حفظ هذه الوثيقة",
    button_style="success",
    layout=widgets.Layout(width="180px")
)
save_all_btn = widgets.Button(
    description="✅ حفظ الكل ← human_gt.json",
    button_style="primary",
    layout=widgets.Layout(width="240px")
)

# ── Helpers ───────────────────────────────────────────────────────────────────
def _null_count():
    total = sum(
        sum(1 for v in d["fields"].values() if v is None)
        for d in human_gt.values()
    )
    return total

def _render_fields(doc_type, existing: dict):
    global _field_widgets
    _field_widgets = {}
    expected = SCHEMA_FIELDS.get(doc_type, [])
    with fields_out:
        clear_output(wait=True)
        if not expected:
            print("لا توجد حقول لهذا النوع في الـ schema")
            return
        rows = []
        for fname in expected:
            val = existing.get(fname)
            txt = "" if val is None else str(val)
            color = "#fff3cd" if val is None else "white"   # yellow = null
            w = widgets.Text(
                value=txt, placeholder="null",
                layout=widgets.Layout(width="380px", background_color=color)
            )
            _field_widgets[fname] = w
            lbl = widgets.HTML(
                f"<div style='width:220px;text-align:right;padding:4px 8px;"
                f"font-size:13px;direction:rtl'>{fname}</div>"
            )
            rows.append(widgets.HBox([lbl, w]))
        display(widgets.VBox(rows))
    null_lbl.value = f"null في هذه الوثيقة: {sum(1 for v in existing.values() if v is None)}"

# ── Event handlers ─────────────────────────────────────────────────────────────
def _on_doc_change(change):
    doc_id = change["new"]
    doc    = human_gt[doc_id]
    type_dd.unobserve(_on_type_change, names="value")
    type_dd.value = doc["doc_type"]
    type_dd.observe(_on_type_change, names="value")
    _render_fields(doc["doc_type"], doc["fields"])
    status_lbl.value = f"تعديل: {doc_id}"

def _on_type_change(change):
    # carry over any matching field values from current widgets before re-rendering
    current_vals = {n: (w.value.strip() or None) for n, w in _field_widgets.items()}
    _render_fields(change["new"], current_vals)

def _on_save_doc(_):
    doc_id   = doc_dd.value
    doc_type = type_dd.value
    fields   = {n: (w.value.strip() or None) for n, w in _field_widgets.items()}
    human_gt[doc_id] = {"doc_type": doc_type, "fields": fields}
    remaining = sum(1 for v in fields.values() if v is None)
    status_lbl.value = f"✓ حُفظت {doc_id} محلياً  |  null متبقية: {remaining}"
    null_lbl.value   = f"null في هذه الوثيقة: {remaining}"

def _on_save_all(_):
    with open(HUMAN_GT_PATH, "w", encoding="utf-8") as f:
        json.dump(human_gt, f, ensure_ascii=False, indent=2)
    total_null = _null_count()
    status_lbl.value = f"✅ حُفظ {len(human_gt)} وثيقة → human_gt.json  |  إجمالي null: {total_null}"

doc_dd.observe(_on_doc_change, names="value")
type_dd.observe(_on_type_change, names="value")
save_doc_btn.on_click(_on_save_doc)
save_all_btn.on_click(_on_save_all)

# ── Initial render ────────────────────────────────────────────────────────────
first = doc_opts[0]
type_dd.value = human_gt[first]["doc_type"]
_render_fields(human_gt[first]["doc_type"], human_gt[first]["fields"])
status_lbl.value = f"تعديل: {first}  |  إجمالي null: {_null_count()}"

display(widgets.VBox([
    widgets.HTML("<h3 style='direction:rtl;margin:8px 0'>محرر Ground Truth البشري</h3>"),
    widgets.HBox([doc_dd, type_dd]),
    widgets.HTML("<hr style='margin:6px 0'>"),
    fields_out,
    widgets.HTML("<hr style='margin:6px 0'>"),
    widgets.HBox([save_doc_btn, save_all_btn]),
    widgets.HBox([status_lbl, widgets.Label("   "), null_lbl]),
]))

In [ ]:
def run_benchmark(provider: str, num_docs: int = 15) -> list:
    if not os.path.exists(GT_PATH):
        raise FileNotFoundError("Run generate_ground_truth() first.")
    with open(GT_PATH, encoding="utf-8") as f:
        gt = json.load(f)

    image_root = f"{data_dir}/processed_images"
    results    = []

    for doc_id in tqdm(sorted(gt.keys())[:num_docs], desc=f"[{provider}]"):
        pages  = sorted(glob(f"{image_root}/{doc_id}/*.jpeg"))
        pred   = run_pipeline(pages, doc_id, provider=provider)
        scores = score_doc(gt[doc_id], pred)
        results.append({
            "doc_id":    doc_id,
            "provider":  provider,
            "gt_type":   gt[doc_id]["doc_type"],
            "pred_type": pred["doc_type"],
            **scores,
            "timing":    pred["timing"],
        })

    return results


def print_summary(results: list):
    n, p = len(results), results[0]["provider"]
    cls  = sum(r["cls_correct"] for r in results)
    f1   = sum(r["f1"] for r in results) / n
    va   = sum(r["value_accuracy"] for r in results) / n
    avgt = sum(r["timing"]["total"] for r in results) / n

    print(f"\n{'='*65}")
    print(f"  {p.upper():<12} Cls={cls}/{n} ({cls/n:.0%})  F1={f1:.3f}  VA={va:.3f}  Avg={avgt:.1f}s")
    print(f"{'='*65}")
    for r in results:
        icon = "✓" if r["cls_correct"] else "✗"
        t    = r["timing"]
        print(f"  {icon} {r['doc_id']}  {r['pred_type']:<28}  "
              f"F1={r['f1']:.2f} VA={r['value_accuracy']:.2f}  "
              f"[{t['stage1']:.1f}+{t['stage2']:.1f}+{t['stage3']:.1f}]s")


def comparison_table(*result_lists):
    """Pass any number of run_benchmark() outputs."""
    header = f"{'Provider':<12} {'Cls Acc':>8} {'Field F1':>9} {'Val Acc':>8} {'Avg Time':>9}"
    print("\n" + header)
    print("-" * len(header))
    for results in result_lists:
        if not results:
            continue
        n, p = len(results), results[0]["provider"]
        cls  = sum(r["cls_correct"] for r in results)
        f1   = sum(r["f1"] for r in results) / n
        va   = sum(r["value_accuracy"] for r in results) / n
        avgt = sum(r["timing"]["total"] for r in results) / n
        print(f"{p:<12} {cls}/{n:>5}   {f1:>8.3f} {va:>8.3f} {avgt:>8.1f}s")

print("Benchmark runner loaded ✓")

## 13 — Run Full Benchmark
Order: smoke test (Cell 9) → ground truth (Cell 10) → benchmarks below.

In [ ]:
# ── Groq ──────────────────────────────────────────────────────────────────────
groq_results = run_benchmark(provider="groq", num_docs=15)
print_summary(groq_results)
with open(f"{data_dir}/results_groq.json", "w", encoding="utf-8") as f:
    json.dump(groq_results, f, ensure_ascii=False, indent=2)

In [ ]:
# ── Gemma3 ────────────────────────────────────────────────────────────────────
gemma_results = run_benchmark(provider="gemma", num_docs=15)
print_summary(gemma_results)
with open(f"{data_dir}/results_gemma.json", "w", encoding="utf-8") as f:
    json.dump(gemma_results, f, ensure_ascii=False, indent=2)

In [ ]:
# ── Final comparison table: Groq vs Gemma3 (Claude = ground truth) ────────────
comparison_table(groq_results, gemma_results)